# Lecture 5: Fitting failure modes lab

**PHYS690: Computational Methods for Physics Research**  
**Thursday, September 10, 2026**

Today is an interactive lab that follows directly from Lecture 04. Instead of introducing many new tools, we will use the same least-squares workflow on examples where fits can go wrong.

We will diagnose:

- label switching in symmetric models,
- true local minima that depend on initial guesses,
- non-convergence and misleading optimizer output,
- physical constraints that help, hurt, or hide problems,
- underconstrained models with large covariance and unstable parameters,
- residual patterns that say the model is not describing the data.

Most of class should be group work. The goal is not to make every fit beautiful. The goal is to learn how to notice when a fit is fragile and how to make a defensible next move.


## How to use this notebook

Run this notebook in VS Code using the course `.venv` kernel. The notebook creates synthetic data and figures under `scratch/lecture05/`, which is intentionally ignored by Git.

Before class, make sure the environment can import the same packages used in Lecture 04:

```bash
source .venv/bin/activate
python -c "import numpy, pandas, scipy, matplotlib; print('ok')"
```

During group activities, keep a short record of symptoms, diagnoses, and fixes. A fit that fails in an informative way is a good result for this lab.


## Learning goals

By the end of this lab, you should be able to:

- Recognize symptoms of a fit failure before trusting parameter values.
- Use residuals, covariance, parameter bounds, parameter labels, and repeated fits to diagnose problems.
- Explain the difference between label-switching degeneracy and a true local minimum.
- Explain how physical constraints can stabilize a fit while also introducing bias if they are too restrictive.
- Identify underconstrained models from large uncertainties, strong correlations, or unstable fitted parameters.
- Document a failed fit clearly enough that someone else can reproduce the diagnosis.


# Part 1: Imports, scratch workspace, and lab helpers

The helper functions below keep the lab focused on diagnosis rather than repeated plotting boilerplate. We will still inspect the model, parameter guesses, bounds, residuals, and covariance matrix for each example.


In [ ]:
%matplotlib inline

from pathlib import Path
import os
import subprocess
import sys
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import OptimizeWarning, curve_fit

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "lectures":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd()

SCRATCH_DIR = PROJECT_ROOT / "scratch" / "lecture05"
DATA_DIR = SCRATCH_DIR / "data"
FIGURE_DIR = SCRATCH_DIR / "figures"

for directory in [DATA_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(seed=6905)

print(f"Python executable: {sys.executable}")
print(f"Working directory: {PROJECT_ROOT}")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")
print(f"SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")


In [ ]:
def poisson_uncertainty(counts):
    """Estimate one-standard-deviation counting uncertainty for Poisson counts."""
    return np.sqrt(np.maximum(counts, 1.0))


def chi_square_from_columns(data, y_column, sigma_column, model_values):
    """Compute chi-square using measured values and uncertainties from a DataFrame."""
    normalized_residuals = (data[y_column] - model_values) / data[sigma_column]
    return np.sum(normalized_residuals**2)


def safe_curve_fit(label, model, data, x_column, y_column, sigma_column, p0, bounds=(-np.inf, np.inf), maxfev=5000):
    """Run curve_fit and return a dictionary even when the fit fails."""
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", OptimizeWarning)
        try:
            popt, pcov = curve_fit(
                model,
                data[x_column],
                data[y_column],
                sigma=data[sigma_column],
                p0=p0,
                bounds=bounds,
                absolute_sigma=True,
                maxfev=maxfev,
            )
            model_values = model(data[x_column], *popt)
            chi2 = chi_square_from_columns(data, y_column, sigma_column, model_values)
            ndf = len(data) - len(popt)
            warning_messages = [str(w.message) for w in caught_warnings]
            return {
                "label": label,
                "success": True,
                "popt": popt,
                "pcov": pcov,
                "chi2": chi2,
                "ndf": ndf,
                "reduced_chi2": chi2 / ndf if ndf > 0 else np.nan,
                "warnings": "; ".join(warning_messages),
                "message": "ok",
            }
        except Exception as error:
            return {
                "label": label,
                "success": False,
                "popt": None,
                "pcov": None,
                "chi2": np.nan,
                "ndf": np.nan,
                "reduced_chi2": np.nan,
                "warnings": "",
                "message": repr(error),
            }


def fit_results_table(results, parameter_names):
    """Convert a list of safe_curve_fit results into a compact DataFrame."""
    rows = []
    for result in results:
        row = {
            "label": result["label"],
            "success": result["success"],
            "chi2": result["chi2"],
            "ndf": result["ndf"],
            "reduced_chi2": result["reduced_chi2"],
            "message": result["message"],
            "warnings": result["warnings"],
        }
        if result["popt"] is not None:
            for name, value in zip(parameter_names, result["popt"]):
                row[name] = value
        rows.append(row)
    return pd.DataFrame(rows)


def covariance_summary(pcov, parameter_names):
    """Return uncertainties and correlation matrix from a covariance matrix."""
    uncertainties = np.sqrt(np.diag(pcov))
    correlation = pcov / np.outer(uncertainties, uncertainties)
    uncertainty_table = pd.DataFrame({"parameter": parameter_names, "uncertainty": uncertainties})
    correlation_table = pd.DataFrame(correlation, index=parameter_names, columns=parameter_names)
    return uncertainty_table, correlation_table



def find_local_minima_2d(values):
    """Return local minima from a 2D grid, excluding the grid boundary."""
    local_minima = []
    for row in range(1, values.shape[0] - 1):
        for col in range(1, values.shape[1] - 1):
            neighborhood = values[row - 1 : row + 2, col - 1 : col + 2]
            if values[row, col] == np.min(neighborhood):
                local_minima.append((values[row, col], row, col))
    return sorted(local_minima, key=lambda item: item[0])


### In-class coding activity 1: environment and helper check, 4 minutes

Run the import and helper cells above. Then run the cell below and make sure the scratch directories exist.

Discuss with your group: what information should you record when a fit fails?


In [ ]:
print("Lecture 05 scratch directory:", SCRATCH_DIR)
print("Data directory exists:", DATA_DIR.exists())
print("Figure directory exists:", FIGURE_DIR.exists())

failure_log = pd.DataFrame(
    columns=["example", "symptom", "likely_cause", "attempted_fix", "what_to_check_next"]
)
failure_log


# Part 2: A failure-mode map for least-squares fitting

Least-squares fitting is powerful, but it is not a truth machine. Use this table as a diagnostic checklist during the lab.

| Symptom | Possible cause | Useful response |
| --- | --- | --- |
| Fit does not converge | Bad initial guess, too few iterations, unstable model function | Try better `p0`, raise `maxfev`, simplify model, check units |
| Fit converges to nonsense | Missing physical constraints, poor model, local minimum | Add justified bounds, inspect residuals, run many initial guesses |
| Many different fits have similar $\chi^2$ | Underconstrained model, parameter degeneracy, or label switching | Fix known parameters, add data, impose an ordering convention, report degeneracy |
| Parameter uncertainty is enormous | Flat direction in $\chi^2$, strong covariance, insufficient data | Inspect `pcov`, correlation matrix, contours |
| Parameter is stuck at a bound | Constraint is active and may dominate the result | Ask whether the bound is physical or hiding model failure |
| Residuals have a pattern | Model misses physics or uncertainty model is wrong | Improve model, add background term, revisit uncertainties |
| Reduced $\chi^2$ is far below 1 | Uncertainties may be overestimated or model is too flexible | Check error bars and degrees of freedom |
| Reduced $\chi^2$ is far above 1 | Model mismatch, underestimated uncertainties, outliers | Inspect residuals; do not trust covariance blindly |


# Part 3: Nuclear/particle example 1: overlapping peaks and label switching

A common nuclear/particle task is fitting peaks in a mass or energy spectrum. Here we simulate two overlapping peaks sitting on a smooth background. This resembles situations such as separating nearby detector response peaks, excited nuclear states, or neighboring resonances.

The generated spectrum is the sum of two Gaussian peaks and a linear background:

$$
N(m) = A_1 \exp\left[-\frac{1}{2}\left(\frac{m-\mu_1}{\sigma_1}\right)^2\right]
+ A_2 \exp\left[-\frac{1}{2}\left(\frac{m-\mu_2}{\sigma_2}\right)^2\right]
+ B_0 + B_1(m - 1.0).
$$

The observed bin counts are then drawn from a Poisson distribution with mean $N(m)$ in each bin. In the first plot, we show the total generating model as well as the two separate Gaussian peak components.

This example is focused on **label switching degeneracy**. If the model treats the two Gaussian peaks symmetrically, then swapping every peak-1 parameter with the corresponding peak-2 parameter gives the same total curve and the same $\chi^2$. That is not a true local minimum. It is the same physical model written with different parameter labels.

A practical repair is to define an ordering convention, such as requiring $\mu_1 < \mu_2$, or to report the physical peaks without attaching meaning to the arbitrary labels returned by the optimizer.


In [ ]:
def two_peak_spectrum_model(mass, amp_1, mean_1, sigma_1, amp_2, mean_2, sigma_2, background_0, background_slope):
    """Two Gaussian peaks plus a linear background."""
    peak_1 = amp_1 * np.exp(-0.5 * ((mass - mean_1) / sigma_1) ** 2)
    peak_2 = amp_2 * np.exp(-0.5 * ((mass - mean_2) / sigma_2) ** 2)
    background = background_0 + background_slope * (mass - 1.0)
    return peak_1 + peak_2 + background

mass_data = pd.DataFrame({"mass_GeV": np.linspace(0.82, 1.22, 90)})
true_peak_parameters = [240.0, 0.965, 0.030, 185.0, 1.045, 0.042, 38.0, -15.0]
(
    true_amp_1,
    true_mean_1,
    true_sigma_1,
    true_amp_2,
    true_mean_2,
    true_sigma_2,
    true_background_0,
    true_background_slope,
) = true_peak_parameters

mass_data["generating_peak_1"] = true_amp_1 * np.exp(
    -0.5 * ((mass_data["mass_GeV"] - true_mean_1) / true_sigma_1) ** 2
)
mass_data["generating_peak_2"] = true_amp_2 * np.exp(
    -0.5 * ((mass_data["mass_GeV"] - true_mean_2) / true_sigma_2) ** 2
)
mass_data["generating_background"] = true_background_0 + true_background_slope * (
    mass_data["mass_GeV"] - 1.0
)
mass_data["expected_counts"] = (
    mass_data["generating_peak_1"]
    + mass_data["generating_peak_2"]
    + mass_data["generating_background"]
)
mass_data["counts"] = rng.poisson(mass_data["expected_counts"])
mass_data["sigma_counts"] = poisson_uncertainty(mass_data["counts"])

mass_csv_path = DATA_DIR / "overlapping_peak_spectrum.csv"
mass_data.to_csv(mass_csv_path, index=False)
mass_data.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    mass_data["mass_GeV"],
    mass_data["counts"],
    yerr=mass_data["sigma_counts"],
    fmt="o",
    capsize=2,
    label="synthetic spectrum",
)
ax.plot(mass_data["mass_GeV"], mass_data["expected_counts"], color="black", label="total generating model")
ax.plot(
    mass_data["mass_GeV"],
    mass_data["generating_peak_1"],
    linestyle="--",
    label="Gaussian peak 1 component",
)
ax.plot(
    mass_data["mass_GeV"],
    mass_data["generating_peak_2"],
    linestyle="--",
    label="Gaussian peak 2 component",
)
ax.plot(
    mass_data["mass_GeV"],
    mass_data["generating_background"],
    linestyle=":",
    label="linear background component",
)
ax.set_xlabel("mass (GeV)")
ax.set_ylabel("counts per bin")
ax.set_title("Overlapping peaks with Poisson counting uncertainty")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

peak_data_path = FIGURE_DIR / "overlapping_peak_data.png"
fig.savefig(peak_data_path, dpi=200, bbox_inches="tight")
print(f"Saved figure to: {peak_data_path}")


## Demonstration: two labels, same fit quality

The next cells fit the same two-Gaussian model twice. The starting guesses intentionally put the peaks in opposite label order. If both fits describe the data equally well but assign the lower-mass and higher-mass structures to different parameter names, that is label switching.


In [ ]:
peak_parameter_names = [
    "amp_1", "mean_1", "sigma_1", "amp_2", "mean_2", "sigma_2", "background_0", "background_slope"
]

peak_lower_bounds = [0.0, 0.88, 0.010, 0.0, 0.88, 0.010, 0.0, -200.0]
peak_upper_bounds = [600.0, 1.16, 0.120, 600.0, 1.16, 0.120, 120.0, 200.0]
peak_bounds = (peak_lower_bounds, peak_upper_bounds)

ordered_peak_guess = [220.0, 0.96, 0.035, 180.0, 1.05, 0.045, 35.0, 0.0]
swapped_peak_guess = [180.0, 1.05, 0.045, 220.0, 0.96, 0.035, 35.0, 0.0]

ordered_peak_result = safe_curve_fit(
    "ordered initial labels",
    two_peak_spectrum_model,
    mass_data,
    "mass_GeV",
    "counts",
    "sigma_counts",
    p0=ordered_peak_guess,
    bounds=peak_bounds,
    maxfev=8000,
)
swapped_peak_result = safe_curve_fit(
    "swapped initial labels",
    two_peak_spectrum_model,
    mass_data,
    "mass_GeV",
    "counts",
    "sigma_counts",
    p0=swapped_peak_guess,
    bounds=peak_bounds,
    maxfev=8000,
)

label_switching_table = fit_results_table(
    [ordered_peak_result, swapped_peak_result],
    peak_parameter_names,
)
label_switching_table


In [ ]:
# Plot the two label-switched solutions and their separate components.
label_switching_results = [ordered_peak_result, swapped_peak_result]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, result in zip(axes, label_switching_results):
    fit_parameters = result["popt"]
    (
        fit_amp_1,
        fit_mean_1,
        fit_sigma_1,
        fit_amp_2,
        fit_mean_2,
        fit_sigma_2,
        fit_background_0,
        fit_background_slope,
    ) = fit_parameters

    fit_component_data = pd.DataFrame({"mass_GeV": mass_data["mass_GeV"]})
    fit_component_data["peak_1"] = fit_amp_1 * np.exp(
        -0.5 * ((fit_component_data["mass_GeV"] - fit_mean_1) / fit_sigma_1) ** 2
    )
    fit_component_data["peak_2"] = fit_amp_2 * np.exp(
        -0.5 * ((fit_component_data["mass_GeV"] - fit_mean_2) / fit_sigma_2) ** 2
    )
    fit_component_data["background"] = fit_background_0 + fit_background_slope * (
        fit_component_data["mass_GeV"] - 1.0
    )
    fit_component_data["total"] = (
        fit_component_data["peak_1"] + fit_component_data["peak_2"] + fit_component_data["background"]
    )

    ax.errorbar(mass_data["mass_GeV"], mass_data["counts"], yerr=mass_data["sigma_counts"], fmt="o", capsize=2, label="data")
    ax.plot(fit_component_data["mass_GeV"], fit_component_data["total"], color="black", label="total fit")
    ax.plot(fit_component_data["mass_GeV"], fit_component_data["peak_1"], linestyle="--", label="peak 1 label")
    ax.plot(fit_component_data["mass_GeV"], fit_component_data["peak_2"], linestyle="--", label="peak 2 label")
    ax.plot(fit_component_data["mass_GeV"], fit_component_data["background"], linestyle=":", label="background")
    ax.set_title(f"{result['label']}\n$\\chi^2$ = {result['chi2']:.2f}")
    ax.set_xlabel("mass (GeV)")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("counts per bin")
axes[1].legend(loc="upper right")
fig.tight_layout()

label_switching_path = FIGURE_DIR / "two_peak_label_switching_components.png"
fig.savefig(label_switching_path, dpi=200, bbox_inches="tight")
print(f"Saved figure to: {label_switching_path}")


### In-class coding activity 2: random starts and label switching, 12 minutes

Work in groups. Complete the random-start scan below.

Tasks:

1. Generate many random initial guesses inside physically plausible ranges.
2. Fit the two-peak spectrum from each starting point.
3. Sort successful fits by $\chi^2$.
4. Compare the best few parameter sets.
5. Decide whether the different parameter labels describe genuinely different curves or the same curve with peak labels swapped.

The bounds below encode physical assumptions: positive yields, positive widths, peak positions inside the plotted mass range, and a background that does not become wildly negative.


In [ ]:
number_of_random_starts = 40
random_peak_results = []

for start_index in range(number_of_random_starts):
    # TODO: Draw one random starting point between peak_lower_bounds and peak_upper_bounds.
    trial_p0 = ...

    trial_result = safe_curve_fit(
        f"random start {start_index:02d}",
        two_peak_spectrum_model,
        mass_data,
        "mass_GeV",
        "counts",
        "sigma_counts",
        p0=trial_p0,
        bounds=peak_bounds,
        maxfev=8000,
    )
    random_peak_results.append(trial_result)

random_peak_table = fit_results_table(random_peak_results, peak_parameter_names)
# TODO: Keep only successful fits and sort from smallest chi-square to largest.
successful_peak_table = ...

successful_peak_table.head(10)


In [ ]:
# Plot the best random-start solution.
# TODO: Select the label of the best fit from successful_peak_table.
best_peak_label = ...

best_peak_result = next(result for result in random_peak_results if result["label"] == best_peak_label)
mass_data["best_random_start_fit"] = two_peak_spectrum_model(
    mass_data["mass_GeV"],
    *best_peak_result["popt"],
)
mass_data["best_random_start_normalized_residual"] = (
    mass_data["counts"] - mass_data["best_random_start_fit"]
) / mass_data["sigma_counts"]

fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].errorbar(mass_data["mass_GeV"], mass_data["counts"], yerr=mass_data["sigma_counts"], fmt="o", capsize=2, label="data")
axes[0].plot(mass_data["mass_GeV"], mass_data["best_random_start_fit"], label="best random-start fit")
axes[0].set_ylabel("counts per bin")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].axhline(0.0, color="0.5", lw=1)
axes[1].errorbar(
    mass_data["mass_GeV"],
    mass_data["best_random_start_normalized_residual"],
    yerr=np.ones(len(mass_data)),
    fmt="o",
    capsize=2,
)
axes[1].set_xlabel("mass (GeV)")
axes[1].set_ylabel("norm. residual")
axes[1].grid(alpha=0.25)
fig.tight_layout()

peak_fit_path = FIGURE_DIR / "overlapping_peak_best_random_start_fit.png"
fig.savefig(peak_fit_path, dpi=200, bbox_inches="tight")
print(f"Saved figure to: {peak_fit_path}")


# Part 4: Nuclear/particle example 3: decay spectrum, constraints, and convergence

A second common nuclear/particle task is extracting a lifetime or attenuation scale from a falling distribution. We simulate a decay-time spectrum with a constant accidental background.

The generated model is an exponential decay signal plus a flat accidental background:

$$
N(t) = N_0 e^{-t/\tau} + B.
$$

Here $N_0$ is the initial signal yield, $\tau$ is the lifetime or attenuation scale, and $B$ is the constant background level. The observed counts in each time bin are drawn from a Poisson distribution with mean $N(t)$.

This example focuses on two related issues:

- **Non-convergence:** the optimizer can fail when the initial guess is far away or the maximum number of evaluations is too small.
- **Physical constraints:** parameters such as event yield, lifetime, and background should be non-negative, but constraints can also bias a result if they are too restrictive.

A bounded fit is not automatically a good fit. Always check whether fitted parameters are pressed against the bounds.


In [ ]:
def decay_with_background_model(time, yield_0, lifetime, background):
    """Exponential decay plus constant accidental background."""
    return yield_0 * np.exp(-time / lifetime) + background

decay_data = pd.DataFrame({"time_us": np.linspace(0.2, 8.0, 42)})
true_decay_parameters = [520.0, 2.10, 7.0]
true_yield_0, true_lifetime, true_background = true_decay_parameters

decay_data["generating_decay_signal"] = true_yield_0 * np.exp(-decay_data["time_us"] / true_lifetime)
decay_data["generating_background"] = true_background
decay_data["expected_counts"] = (
    decay_data["generating_decay_signal"] + decay_data["generating_background"]
)
decay_data["counts"] = rng.poisson(decay_data["expected_counts"])
decay_data["sigma_counts"] = poisson_uncertainty(decay_data["counts"])

decay_csv_path = DATA_DIR / "decay_with_background.csv"
decay_data.to_csv(decay_csv_path, index=False)
decay_data.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    decay_data["time_us"],
    decay_data["counts"],
    yerr=decay_data["sigma_counts"],
    fmt="o",
    capsize=2,
    label="synthetic decay spectrum",
)
ax.plot(
    decay_data["time_us"],
    decay_data["expected_counts"],
    color="black",
    label="total generating model",
)
ax.plot(
    decay_data["time_us"],
    decay_data["generating_decay_signal"],
    linestyle="--",
    label="exponential signal component",
)
ax.plot(
    decay_data["time_us"],
    decay_data["generating_background"],
    linestyle=":",
    label="constant background component",
)
ax.set_xlabel("time (microseconds)")
ax.set_ylabel("counts per bin")
ax.set_title("Decay spectrum with accidental background")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()


## Demonstration: force a non-convergent fit

The next fit deliberately uses a poor initial guess and too small a `maxfev`. This is not a recommended analysis choice; it is a controlled way to see what a failed optimization looks like.


In [ ]:
decay_parameter_names = ["yield_0", "lifetime", "background"]

nonconvergent_decay_result = safe_curve_fit(
    "bad guess and too few evaluations",
    decay_with_background_model,
    decay_data,
    "time_us",
    "counts",
    "sigma_counts",
    p0=[50_000.0, 0.02, 500.0],
    maxfev=5,
)

fit_results_table([nonconvergent_decay_result], decay_parameter_names)


### In-class coding activity 3: constraints that help and constraints that hurt, 14 minutes

Work in groups. Compare three fits:

1. an unconstrained fit,
2. a fit with sensible physical constraints,
3. a fit with an over-restrictive constraint.

For each fit, ask:

- Did it converge?
- Is the reduced $\chi^2$ reasonable?
- Are any parameters stuck near a bound?
- Do residuals show a pattern?
- Did the constraint encode real physics or force the answer?


In [ ]:
unconstrained_decay_result = safe_curve_fit(
    "unconstrained",
    decay_with_background_model,
    decay_data,
    "time_us",
    "counts",
    "sigma_counts",
    p0=[400.0, 1.0, 2.0],
    maxfev=5000,
)

# TODO: Choose sensible lower and upper bounds for yield, lifetime, and background.
sensible_decay_bounds = (..., ...)

sensible_decay_result = safe_curve_fit(
    "sensible physical bounds",
    decay_with_background_model,
    decay_data,
    "time_us",
    "counts",
    "sigma_counts",
    p0=[400.0, 1.5, 5.0],
    bounds=sensible_decay_bounds,
    maxfev=5000,
)

# This deliberately over-restrictive bound forces the background above a value that may not match the data.
over_restrictive_decay_bounds = ([0.0, 0.1, 25.0], [2000.0, 10.0, 100.0])
over_restrictive_decay_result = safe_curve_fit(
    "over-restrictive background bound",
    decay_with_background_model,
    decay_data,
    "time_us",
    "counts",
    "sigma_counts",
    p0=[400.0, 1.5, 30.0],
    bounds=over_restrictive_decay_bounds,
    maxfev=5000,
)

decay_fit_table = fit_results_table(
    [unconstrained_decay_result, sensible_decay_result, over_restrictive_decay_result],
    decay_parameter_names,
)
decay_fit_table


In [ ]:
# TODO: Choose which decay fit you think is most defensible.
best_decay_label = ...

best_decay_result = next(
    result for result in [unconstrained_decay_result, sensible_decay_result, over_restrictive_decay_result]
    if result["label"] == best_decay_label
)

decay_data["selected_fit"] = decay_with_background_model(decay_data["time_us"], *best_decay_result["popt"])
decay_data["selected_normalized_residual"] = (
    decay_data["counts"] - decay_data["selected_fit"]
) / decay_data["sigma_counts"]

fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].errorbar(decay_data["time_us"], decay_data["counts"], yerr=decay_data["sigma_counts"], fmt="o", capsize=2, label="data")
axes[0].plot(decay_data["time_us"], decay_data["selected_fit"], label=best_decay_label)
axes[0].set_ylabel("counts per bin")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].axhline(0.0, color="0.5", lw=1)
axes[1].errorbar(decay_data["time_us"], decay_data["selected_normalized_residual"], yerr=np.ones(len(decay_data)), fmt="o", capsize=2)
axes[1].set_xlabel("time (microseconds)")
axes[1].set_ylabel("norm. residual")
axes[1].grid(alpha=0.25)
fig.tight_layout()


# Part 5: Quantum optics example: underconstrained damped oscillations

In quantum optics and quantum information experiments, an observed population can oscillate as a function of pulse duration or delay time. A simple damped oscillation model is

$$
P(t) = P_0 + C e^{-t/T_2} \cos(2\pi f t + \phi).
$$

This looks innocent, but it can be difficult to fit if the time range is short, the contrast is small, or too many parameters are allowed to vary at once.

The key identifiability issue here is the coherence time $T_2$. The synthetic data cover only a short window, from 0 to 1.8 microseconds, while the true coherence time is 8 microseconds. Over such a short window the damping envelope changes slowly, so many large values of $T_2$ produce nearly the same curve. That makes $T_2$ weakly identified.

This example exposes underconstrained-model behavior:

- $T_2$ can run to a bound even when the fit converges,
- the $T_2$ uncertainty can be much larger than the fitted value,
- contrast and $T_2$ can be strongly anticorrelated,
- frequency and phase can be strongly correlated because a short time window cannot cleanly separate them,
- a reduced model with a fixed $T_2$ may be more stable if that constraint comes from independent calibration.


In [ ]:
def damped_oscillation_probability(time, offset, contrast, frequency_MHz, phase, coherence_time_us):
    """Damped oscillation model for a measured excited-state probability."""
    return offset + contrast * np.exp(-time / coherence_time_us) * np.cos(2.0 * np.pi * frequency_MHz * time + phase)

quantum_data = pd.DataFrame({"time_us": np.linspace(0.0, 1.8, 24)})
true_quantum_parameters = [0.50, 0.22, 0.82, 0.45, 8.0]
quantum_data["true_probability"] = damped_oscillation_probability(quantum_data["time_us"], *true_quantum_parameters)
quantum_data["true_probability"] = np.clip(quantum_data["true_probability"], 0.02, 0.98)
quantum_data["shots"] = 180
quantum_data["excited_counts"] = rng.binomial(quantum_data["shots"], quantum_data["true_probability"])
quantum_data["probability"] = quantum_data["excited_counts"] / quantum_data["shots"]
quantum_data["sigma_probability"] = np.sqrt(
    np.maximum(quantum_data["probability"] * (1.0 - quantum_data["probability"]) / quantum_data["shots"], 1.0 / quantum_data["shots"]**2)
)

quantum_csv_path = DATA_DIR / "underconstrained_quantum_oscillation.csv"
quantum_data.to_csv(quantum_csv_path, index=False)
quantum_data.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    quantum_data["time_us"],
    quantum_data["probability"],
    yerr=quantum_data["sigma_probability"],
    fmt="o",
    capsize=3,
    label="synthetic quantum optics data",
)
ax.plot(quantum_data["time_us"], quantum_data["true_probability"], label="generating model")
ax.set_xlabel("time (microseconds)")
ax.set_ylabel("excited-state probability")
ax.set_title("Short-window damped oscillation data")
ax.set_ylim(0.0, 1.0)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()


### In-class coding activity 4: diagnose an underconstrained model, 20 minutes

Work in groups, but start from the provided full five-parameter fit below. The purpose of this first fit is diagnostic: it shows that an optimizer can converge while one parameter is still not meaningfully determined by the data.

As you run the next few cells, look for four symptoms:

1. Is `coherence_time_us` pushed close to the upper bound?
2. Is its uncertainty much larger than the fitted value?
3. Which parameter pairs have correlations closest to $+1$ or $-1$?
4. Would the residual plot alone warn you that $T_2$ is weakly identified?


In [ ]:
quantum_parameter_names = ["offset", "contrast", "frequency_MHz", "phase", "coherence_time_us"]

# Bounds keep probabilities and physical scales in a plausible range.
quantum_bounds = ([0.0, 0.0, 0.1, -np.pi, 0.2], [1.0, 0.5, 2.0, np.pi, 50.0])

# Start near a reasonable oscillation, but do not assume the data can determine every parameter.
quantum_initial_guess = [0.50, 0.20, 0.80, 0.30, 10.0]

full_quantum_result = safe_curve_fit(
    "full five-parameter model",
    damped_oscillation_probability,
    quantum_data,
    "time_us",
    "probability",
    "sigma_probability",
    p0=quantum_initial_guess,
    bounds=quantum_bounds,
    maxfev=10000,
)

full_quantum_fit_table = fit_results_table([full_quantum_result], quantum_parameter_names)
full_quantum_fit_table


The fit table tells you whether SciPy found a numerical optimum. It does not, by itself, tell you whether all parameters are identifiable. The next diagnostic is the covariance matrix: large uncertainties and strong correlations are signs that the local $\chi^2$ surface has a shallow direction.


In [ ]:
if full_quantum_result["success"]:
    quantum_uncertainties, quantum_correlation = covariance_summary(
        full_quantum_result["pcov"],
        quantum_parameter_names,
    )
    display(quantum_uncertainties)
    display(quantum_correlation)

    # Plot the correlation matrix so the strongest parameter tradeoffs are visible.
    fig, ax = plt.subplots(figsize=(6, 5))
    correlation_image = ax.imshow(
        quantum_correlation,
        vmin=-1.0,
        vmax=1.0,
        cmap="coolwarm",
    )
    ax.set_xticks(np.arange(len(quantum_parameter_names)))
    ax.set_yticks(np.arange(len(quantum_parameter_names)))
    ax.set_xticklabels(quantum_parameter_names, rotation=45, ha="right")
    ax.set_yticklabels(quantum_parameter_names)

    for row_index in range(len(quantum_parameter_names)):
        for col_index in range(len(quantum_parameter_names)):
            ax.text(
                col_index,
                row_index,
                f"{quantum_correlation.iloc[row_index, col_index]:.2f}",
                ha="center",
                va="center",
                color="black",
                fontsize=8,
            )

    ax.set_title("Full quantum fit parameter correlation matrix")
    fig.colorbar(correlation_image, ax=ax, label="correlation coefficient")
    fig.tight_layout()

    quantum_correlation_path = FIGURE_DIR / "quantum_fit_correlation_matrix.png"
    fig.savefig(quantum_correlation_path, dpi=200, bbox_inches="tight")
    print(f"Saved figure to: {quantum_correlation_path}")
else:
    print(full_quantum_result["message"])


Now convert the covariance output into a short diagnostic table. For this data set, the strongest warning sign should be the weakly identified coherence time: the fitted value often sits near the upper bound, and the uncertainty can be far larger than the estimate itself.

Two correlations are especially important to interpret physically:

- **contrast and $T_2$:** a longer coherence time preserves oscillation contrast, so the initial contrast can compensate in the opposite direction.
- **frequency and phase:** over a short time window, a small frequency change can look like a phase shift, so these parameters can trade off.


In [ ]:
if full_quantum_result["success"]:
    full_quantum_parameter_table = full_quantum_fit_table.melt(
        id_vars=["label", "success", "chi2", "ndf", "reduced_chi2", "message", "warnings"],
        value_vars=quantum_parameter_names,
        var_name="parameter",
        value_name="estimate",
    ).merge(quantum_uncertainties, on="parameter")

    lower_bounds, upper_bounds = quantum_bounds
    bounds_table = pd.DataFrame(
        {
            "parameter": quantum_parameter_names,
            "lower_bound": lower_bounds,
            "upper_bound": upper_bounds,
        }
    )
    full_quantum_parameter_table = full_quantum_parameter_table.merge(bounds_table, on="parameter")
    full_quantum_parameter_table["uncertainty / abs(estimate)"] = (
        full_quantum_parameter_table["uncertainty"] / np.abs(full_quantum_parameter_table["estimate"])
    )
    full_quantum_parameter_table["distance_to_upper_bound"] = (
        full_quantum_parameter_table["upper_bound"] - full_quantum_parameter_table["estimate"]
    )

    correlation_pairs = []
    for row_index, row_name in enumerate(quantum_parameter_names):
        for col_index, col_name in enumerate(quantum_parameter_names):
            if col_index <= row_index:
                continue
            correlation_pairs.append(
                {
                    "parameter_1": row_name,
                    "parameter_2": col_name,
                    "correlation": quantum_correlation.loc[row_name, col_name],
                    "abs_correlation": abs(quantum_correlation.loc[row_name, col_name]),
                }
            )
    strongest_correlations = (
        pd.DataFrame(correlation_pairs)
        .sort_values("abs_correlation", ascending=False)
        .reset_index(drop=True)
    )

    display(full_quantum_parameter_table)
    display(strongest_correlations.head(5))
else:
    print(full_quantum_result["message"])


The next comparison asks whether a physically motivated constraint helps. If an independent calibration already measured $T_2$, fixing it can remove the weakly identified direction from the fit. But this is only defensible if the fixed value comes from physics or calibration information outside this fit. Fixing a parameter just to make the covariance matrix look better is not a scientific solution.


In [ ]:
def fixed_decay_oscillation_model(time, offset, contrast, frequency_MHz, phase):
    """Damped oscillation with coherence time fixed from independent calibration."""
    fixed_coherence_time_us = 8.0
    return damped_oscillation_probability(time, offset, contrast, frequency_MHz, phase, fixed_coherence_time_us)

reduced_quantum_parameter_names = ["offset", "contrast", "frequency_MHz", "phase"]

# TODO: Choose a starting point and bounds for the reduced model.
reduced_quantum_initial_guess = ...
reduced_quantum_bounds = (..., ...)

reduced_quantum_result = safe_curve_fit(
    "fixed coherence-time model",
    fixed_decay_oscillation_model,
    quantum_data,
    "time_us",
    "probability",
    "sigma_probability",
    p0=reduced_quantum_initial_guess,
    bounds=reduced_quantum_bounds,
    maxfev=10000,
)

full_summary = fit_results_table([full_quantum_result], quantum_parameter_names)
reduced_summary = fit_results_table([reduced_quantum_result], reduced_quantum_parameter_names)
fit_model_comparison = pd.concat([full_summary, reduced_summary], ignore_index=True, sort=False)
fit_model_comparison


Finally choose which model to plot. The residuals help answer a different question from the covariance matrix. Residuals ask whether the curve describes the measured points. The covariance matrix asks whether the data determine the fitted parameters. A model can have visually acceptable residuals while still having a weakly identified parameter.


In [ ]:
# TODO: Choose which quantum model your group wants to plot.
selected_quantum_label = ...

selected_quantum_result = next(
    result for result in [full_quantum_result, reduced_quantum_result]
    if result["label"] == selected_quantum_label
)

smooth_quantum_data = pd.DataFrame(
    {"time_us": np.linspace(quantum_data["time_us"].min(), quantum_data["time_us"].max(), 300)}
)

if selected_quantum_label == "full five-parameter model":
    smooth_quantum_data["selected_fit"] = damped_oscillation_probability(
        smooth_quantum_data["time_us"],
        *selected_quantum_result["popt"],
    )
    quantum_data["selected_fit"] = damped_oscillation_probability(
        quantum_data["time_us"],
        *selected_quantum_result["popt"],
    )
else:
    smooth_quantum_data["selected_fit"] = fixed_decay_oscillation_model(
        smooth_quantum_data["time_us"],
        *selected_quantum_result["popt"],
    )
    quantum_data["selected_fit"] = fixed_decay_oscillation_model(
        quantum_data["time_us"],
        *selected_quantum_result["popt"],
    )

quantum_data["selected_normalized_residual"] = (
    quantum_data["probability"] - quantum_data["selected_fit"]
) / quantum_data["sigma_probability"]

fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].errorbar(quantum_data["time_us"], quantum_data["probability"], yerr=quantum_data["sigma_probability"], fmt="o", capsize=3, label="data")
axes[0].plot(smooth_quantum_data["time_us"], smooth_quantum_data["selected_fit"], label=selected_quantum_label)
axes[0].set_ylabel("probability")
axes[0].set_ylim(0.0, 1.0)
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].axhline(0.0, color="0.5", lw=1)
axes[1].errorbar(quantum_data["time_us"], quantum_data["selected_normalized_residual"], yerr=np.ones(len(quantum_data)), fmt="o", capsize=3)
axes[1].set_xlabel("time (microseconds)")
axes[1].set_ylabel("norm. residual")
axes[1].grid(alpha=0.25)
fig.tight_layout()


## Optional Git/GitHub checkpoint

If time permits, save the notebook and commit your lab notes on your Lecture 05 branch.

```bash
git status
git switch -c your-username-lecture05
git add lectures/lecture05_9_10_26_fit_failure_modes_lab.ipynb
git commit -m "Complete Lecture 05 fitting failure modes lab"
git push -u origin your-username-lecture05
git status
```

If the branch already exists, use `git switch your-username-lecture05`.


# Part 6: What to carry into assignments and projects

When you report a fit, include more than the parameter table.

A defensible fit report should include:

| Item | Why it matters |
| --- | --- |
| Data with error bars | Shows what was actually fit and how points were weighted |
| Model equation and assumptions | Clarifies what physics was included or omitted |
| Initial guesses and bounds | Makes nonlinear fitting reproducible |
| Best-fit parameters with uncertainties | Gives the numerical result and scale of uncertainty |
| Covariance or correlation matrix | Reveals parameter degeneracy |
| Residual plot | Shows systematic disagreement that $\chi^2$ alone can hide |
| Convergence and failure checks | Documents whether the optimizer result is stable |
| Alternative attempts | Shows that you looked for local minima, label switching, and modeling traps |

The scientific question is not "Did SciPy return numbers?" The scientific question is "What evidence do I have that these numbers mean what I claim they mean?"
